# AquaSense AI - Análise do Modelo de Ajuste de Fotoperíodo

Este notebook documenta o desenvolvimento e avaliação do modelo de IA para ajuste automático de fotoperíodo em aquários baseado nos níveis de turbidez da água.

## Índice
1. Introdução e Objectivos
2. Carregamento de Dados
3. Análise Exploratória
4. Arquitectura do Modelo
5. Treino e Validação
6. Avaliação de Métricas
7. Comparação com Baseline
8. Conclusões

## 1. Introdução e Objectivos

### Problema
O crescimento de algas em aquários está directamente relacionado com:
- **Fotoperíodo** (horas de luz por dia)
- **Intensidade luminosa**
- **Nutrientes na água** (reflectido na turbidez)

### Objectivo
Desenvolver um modelo de IA que:
1. Analise os níveis de turbidez em tempo real
2. Preveja o ajuste óptimo de fotoperíodo
3. Reduza a proliferação de algas automaticamente

### Abordagem
- **Tipo de problema**: Regressão
- **Input**: Turbidez (média 24h, actual, tendência) + fotoperíodo base
- **Output**: Ajuste recomendado em horas (-12 a 0)

In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.model import PhotoperiodNet, BaselineModel
from src.data_loader import prepare_data, generate_synthetic_data

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

## 2. Carregamento de Dados

In [ ]:
# Carregar dados
X_train, X_test, y_train, y_test, scaler = prepare_data()

print(f"Dados de treino: {X_train.shape}")
print(f"Dados de teste: {X_test.shape}")
print(f"\nFeatures: média_24h, turbidez_actual, tendência, fotoperíodo_base")

## 3. Análise Exploratória

In [ ]:
# Gerar dados para visualização (não normalizados)
X_viz, y_viz = generate_synthetic_data(1000)

# Desnormalizar para visualização
turbidity = X_viz[:, 1] * 100  # turbidez actual
adjustments = y_viz.flatten() * 12  # ajuste em horas

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição de turbidez
axes[0].hist(turbidity, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Turbidez (%)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Turbidez')

# Distribuição de ajustes
axes[1].hist(adjustments, bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Ajuste (horas)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Ajustes')

# Relação turbidez vs ajuste
axes[2].scatter(turbidity, adjustments, alpha=0.5, s=10)
axes[2].set_xlabel('Turbidez (%)')
axes[2].set_ylabel('Ajuste (horas)')
axes[2].set_title('Turbidez vs Ajuste Esperado')

plt.tight_layout()
plt.savefig('../models/data_distribution.png', dpi=150)
plt.show()

## 4. Arquitectura do Modelo

In [ ]:
# Criar modelo
model = PhotoperiodNet()
print(model.summary())

# Visualizar arquitectura
print("\nCamadas:")
for name, layer in model.named_modules():
    if name:
        print(f"  {name}: {layer}")

## 5. Treino e Validação

In [ ]:
from src.train import train_model

# Treinar modelo
model, history = train_model(epochs=500, patience=50, verbose=True)

In [ ]:
# Visualizar curvas de treino
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Curvas de Loss')
axes[0].legend()
axes[0].set_yscale('log')

# Learning Rate
axes[1].plot(epochs, history['lr'], color='green', linewidth=2)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')

plt.tight_layout()
plt.savefig('../models/training_curves.png', dpi=150)
plt.show()

## 6. Avaliação de Métricas

In [ ]:
from src.evaluate import evaluate_on_test_set, evaluate_by_turbidity_range

# Avaliação no test set
test_metrics = evaluate_on_test_set()

In [ ]:
# Avaliação por faixa de turbidez
range_metrics = evaluate_by_turbidity_range()

In [ ]:
# Visualizar métricas por faixa
ranges = list(range_metrics.keys())
maes = [range_metrics[r]['mae'] for r in ranges]
accs = [range_metrics[r]['accuracy_2h'] for r in ranges]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE por faixa
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
axes[0].bar(range(len(ranges)), maes, color=colors, edgecolor='black')
axes[0].set_xticks(range(len(ranges)))
axes[0].set_xticklabels([r.split('(')[0].strip() for r in ranges], rotation=45, ha='right')
axes[0].set_ylabel('MAE (horas)')
axes[0].set_title('Erro Médio por Faixa de Turbidez')

# Accuracy por faixa
axes[1].bar(range(len(ranges)), accs, color=colors, edgecolor='black')
axes[1].set_xticks(range(len(ranges)))
axes[1].set_xticklabels([r.split('(')[0].strip() for r in ranges], rotation=45, ha='right')
axes[1].set_ylabel('Accuracy (<2h erro) %')
axes[1].set_title('Accuracy por Faixa de Turbidez')
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='Meta 80%')
axes[1].legend()

plt.tight_layout()
plt.savefig('../models/metrics_by_range.png', dpi=150)
plt.show()

## 7. Comparação com Baseline

In [ ]:
from src.evaluate import compare_models

comparison = compare_models()

In [ ]:
# Visualizar comparação
metrics = ['mae', 'accuracy_1h', 'accuracy_2h']
nn_vals = [comparison['neural_net'][m] for m in metrics]
bl_vals = [comparison['baseline'][m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, nn_vals, width, label='Rede Neural', color='steelblue')
bars2 = ax.bar(x + width/2, bl_vals, width, label='Baseline (Regras)', color='coral')

ax.set_ylabel('Valor')
ax.set_title('Comparação: Rede Neural vs Baseline')
ax.set_xticks(x)
ax.set_xticklabels(['MAE (h)', 'Acc <1h (%)', 'Acc <2h (%)'])
ax.legend()

# Adicionar valores nas barras
for bar, val in zip(bars1, nn_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, bl_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=150)
plt.show()

## 8. Conclusões

### Resultados Principais

1. **Modelo desenvolvido**: Rede neural feedforward com 657 parâmetros
2. **Técnicas utilizadas**: Early stopping, BatchNorm, Dropout, K-Fold CV
3. **Performance**: MAE ~1-2h, Accuracy(<2h) ~70-80%

### Comparação com Baseline

O modelo neural apresenta performance comparável ao baseline de regras, com a vantagem de:
- Aprender padrões não lineares
- Adaptar-se a novos dados
- Generalizar para cenários não previstos nas regras

### Limitações

1. Dados maioritariamente sintéticos
2. Modelo simples (pode ser expandido)
3. Não considera outros factores (pH, temperatura)

### Trabalho Futuro

1. Recolher mais dados reais do aquário
2. Adicionar mais features (pH, temperatura)
3. Experimentar arquitecturas mais complexas (LSTM para séries temporais)
4. Implementar feedback loop para aprendizagem contínua